<a href="https://colab.research.google.com/github/FedeBeraldo/dm2026b/blob/desktop-jr/914_multiples_semillas_ia_adivina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,667663,35.7,1479497,79.1,1259893,67.3
Vcells,1234504,9.5,8388608,64.0,1978697,15.1


In [2]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

In [3]:
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

Loading required package: lightgbm



In [4]:
PARAM <- list()
PARAM$experimento <- 912
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WFA", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))
dataset_original <- fread(paste0("/content/datasets/", PARAM$dataset))

In [5]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [6]:
# 1) Catastrophe Analysis

# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}

Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}


Corregir_atributo <- function(pcampo, pmeses, pmetodo){
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}


Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}

# 2) DR Data Drifting
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)

# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)

tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}

drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}

drift_deflacion <- function(dataset, campos_monetarios) {

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  return (dataset)
}

drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}

# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(dataset, campos_drift) {

  cat("inicio drift_rank_cero_fijo()\n")

  for (campo in campos_drift)
  {
    cat(campo, " ")

    dataset[
      get(campo) == 0,
      paste0(campo, "_rank") := 0
    ]

    dataset[
      get(campo) > 0,
      paste0(campo, "_rank") :=
        frank(get(campo), ties.method = "random") / .N,
      by = list(foto_mes)
    ]

    dataset[
      get(campo) < 0,
      paste0(campo, "_rank") :=
        -frank(-get(campo), ties.method = "random") / .N,
      by = list(foto_mes)
    ]

    dataset[, (campo) := NULL]
  }

  cat("\n")
  cat("fin drift_rank_cero_fijo()\n")

  return(dataset)
}

drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}

# 3) FE_intra_manual Feature Engineering intra-mes

# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function(dataset, patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# Adaptación de Campos IA por función rank_cero_fijo

adaptar_campos_ia_rank <- function(campos_ia, campos_monetarios) {

  sapply(campos_ia, function(campo) {

    # ¿Es una variable derivada de una variable monetaria?
    for (monetaria in campos_monetarios) {

      # Variable original
      if (campo == monetaria) {
        return(paste0(monetaria, "_rank"))
      }

      # Lag
      if (campo == paste0(monetaria, "_lag1")) {
        return(paste0(monetaria, "_rank_lag1"))
      }

      if (campo == paste0(monetaria, "_lag2")) {
        return(paste0(monetaria, "_rank_lag2"))
      }

      # Delta
      if (campo == paste0(monetaria, "_delta1")) {
        return(paste0(monetaria, "_rank_delta1"))
      }

      if (campo == paste0(monetaria, "_delta2")) {
        return(paste0(monetaria, "_rank_delta2"))
      }
    }

    # Si no es monetaria, queda igual
    return(campo)
  }, USE.NAMES = FALSE)
}


In [9]:

########################################################################################
###               PARA AGREGAR "IA FEATURE SELECTION" AL WORKFLOW JUNIOR             ###
### DENTRO DE LA FUNCIÓN, COPIAR SECCION 6: "FEATURE SELECTION BASADA EN IA EXPERTA" ###
###                           - Se agrega entre secciones 5) y 7)                    ###
########################################################################################

run_experimento <- function(semilla) {

    PARAM$semilla_primigenia <- semilla
    dataset <- copy(dataset_original)  # Recupero el original para canaritos nuevos
    # =================================
    # TODO EL WORKFLOW ACTUAL
    # =================================

    cat(
        format(Sys.time()),
        " INICIO SEMILLA ",
        semilla,
        "\n"
    )
    flush.console()

    #1. resuelvo el Catastrophe Analysis

    setorder( dataset, numero_de_cliente, foto_mes )

    PARAM$CA$metodo= "MachineLearning"

    if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
      Corregir_Rotas(dataset, PARAM$CA$metodo)


    #2. ejecuto el Data Drifting
    setorder( dataset, numero_de_cliente, foto_mes )

    # por como armé los nombres de campos,
    #  estos son los campos que expresan variables monetarias
    campos_monetarios <- colnames(dataset)
    campos_monetarios <- campos_monetarios[campos_monetarios %like%
      "^(m|Visa_m|Master_m|vm_m)"]
    #PARAM$DR$metodo <- "deflacion"
    # MOdifiación:
    # Variables Monetarias en Rank ({original}_rank):
    # Mantienen la misma capacidad predictiva para detectar abandono.
    #El mecanismo subyacente de abandono no depende del monto en pesos absolutos, sino de si el cliente está descendiendo en la distribución de riqueza/actividad del banco.
    # Los delta también se vuelven más significativos!!
    # https://gemini.google.com/app/5556782dcf026d26
    
    PARAM$DR$metodo <- "rank_cero_fijo"

    #switch(PARAM$DR$metodo,
    #  "ninguno"        = cat("No hay correccion del data drifting"),
    #  "rank_simple"    = drift_rank_simple(campos_monetarios),
    #  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
    #  "deflacion"      = drift_deflacion(campos_monetarios),
    #  "dolar_blue"     = drift_dolarblue(campos_monetarios),
    #  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
    #  "UVA"            = drift_UVA(campos_monetarios),
    #  "estandarizar"   = drift_estandarizar(campos_monetarios)
    #)

    # MODIFICADO: aplico drift_rank, esto a su vez me dispara otros cambios
    dataset <- drift_rank_cero_fijo(

        dataset,

        campos_monetarios

    )

    # 3) FE_intra_manual Feature Engineering intra-mes

    # el mes 1,2, ..12
    if( atributos_presentes(dataset, c("foto_mes") ))
      dataset[, kmes := foto_mes %% 100]

    # variable extraida de una tesis de maestria de Irlanda

    #MODIFICADO: Reemplazo por el payroll por el rank payroll
    #Gemini: mpayroll_rank / cliente_edad: Sigue siendo de altísimo interés. 
    #Al sustituir mpayroll absoluto por mpayroll_rank, la variable mide la capacidad relativa de ingreso normalizada por ciclo de vida.
    #Representa qué tan elevado es el poder adquisitivo del cliente en comparación con sus pares del mismo mes, ajustado por su edad.

    if( atributos_presentes(dataset, c("mpayroll_rank", "cliente_edad") ))
      dataset[, mpayroll_sobre_edad := mpayroll_rank / cliente_edad]

    # 4) No se implementa en junior

    # 5) Feature Engineering Historico

    # todo es lagueable, menos la primary key y la clase

    cols_lagueables <- copy( setdiff(
        colnames(dataset),
        c("numero_de_cliente", "foto_mes", "clase_ternaria")
    ) )

    # https://rdrr.io/cran/data.table/man/shift.html
    # lags de orden 1
    dataset[,
        paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
        by = numero_de_cliente,
        .SDcols = cols_lagueables
    ]

    # lags de orden 2
    dataset[,
        paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
        by = numero_de_cliente,
        .SDcols = cols_lagueables
    ]

    # agrego los delta lags
    for (vcol in cols_lagueables)
    {
        dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
        dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
    }

    # ----------------------------------------------------------
    # PREPARACIÓN
    # ----------------------------------------------------------

    PARAM$trainingstrategy$validate <- c(202107)

    PARAM$trainingstrategy$training <- c(
      201901, 201902, 201903, 201904, 201905, 201906,
      201907, 201908, 201909, 201910, 201911, 201912,
      202001, 202002, 202101, 202102, 202103, 202104,
      202105
    )

    PARAM$trainingstrategy$training_pct <- 1.0
    PARAM$trainingstrategy$positivos <- c("BAJA+1", "BAJA+2")


    # ----------------------------------------------------------
    # clase01
    # ----------------------------------------------------------

    dataset[
      ,
      clase01 := ifelse(
        clase_ternaria %in% PARAM$trainingstrategy$positivos,
        1,
        0
      )
    ]


    # ----------------------------------------------------------
    # azar + undersampling
    # ----------------------------------------------------------

    set.seed(
      PARAM$semilla_primigenia,
      kind = "L'Ecuyer-CMRG"
    )

    dataset[, azar := runif(nrow(dataset))]

    dataset[
      ,
      fold_train :=
        foto_mes %in% PARAM$trainingstrategy$training &
        (
          clase_ternaria %in% c("BAJA+1", "BAJA+2") |
          azar < PARAM$trainingstrategy$training_pct
        )
    ]

    # 6) IA a la obra...

    # ============================================================
    # FEATURE SELECTION BASADA EN IA EXPERTA
    # ============================================================

    campos_ia <- c(
      # Actividad Operativa y Canales
      "ctrx_quarter",
      "ctrx_quarter_lag1",
      "ctrx_quarter_delta1",
      "ctrx_quarter_delta2",
      "chomebanking_transacciones",
      "chomebanking_transacciones_delta1",
      "thomebanking",
      "thomebanking_delta1",
      "ccallcenter_transacciones_delta1",

      # Payroll / Sueldo
      "cpayroll_trx",
      "cpayroll_trx_delta1",
      "mpayroll",
      "mpayroll_delta1",
      "mpayroll_delta2",
      "mpayroll_sobre_edad",

      # Tarjetas
      "ctarjeta_visa",
      "ctarjeta_visa_transacciones",
      "ctarjeta_visa_transacciones_delta1",
      "mtarjeta_visa_consumo",
      "mtarjeta_visa_consumo_delta1",
      "mtarjeta_visa_consumo_delta2",
      "Visa_status",
      "Visa_msaldototal",
      "Visa_msaldototal_delta1",
      "Visa_mpagominimo_delta1",
      "mttarjeta_visa_debitos_automaticos",
      "mttarjeta_visa_debitos_automaticos_delta1",
      "mttarjeta_visa_debitos_automaticos_delta2",
      "ctarjeta_master",
      "ctarjeta_master_transacciones_delta1",
      "mtarjeta_master_consumo_delta1",
      "Master_status",

      # Saldos, cuentas y límites
      "mcuentas_saldo",
      "mcuentas_saldo_lag1",
      "mcuentas_saldo_delta1",
      "mcuentas_saldo_delta2",
      "mcaja_ahorro",
      "mcaja_ahorro_delta1",
      "mcuenta_corriente_delta1",
      "cdescubierto_preacordado",
      "cdescubierto_preacordado_delta1",
      "mtransferencias_recibidas",
      "mtransferencias_recibidas_delta1",

      # Productos, rentabilidad y comisiones
      "cproductos",
      "cproductos_delta1",
      "cproductos_delta2",
      "mrentabilidad_annual",
      "mrentabilidad_annual_delta1",
      "mrentabilidad_annual_delta2",
      "mcomisiones_mantenimiento",
      "ccomisiones_mantenimiento_delta1",
      "ccomisiones_mantenimiento_delta2",
      "mprestamos_personales",
      "cprestamos_personales_delta1",
      "mactivos_margen_delta1",
      "mpasivos_margen_delta1",

      # Perfil
      "cliente_edad",
      "cliente_antiguedad"
    )

    

    #campos_buenos <- copy(campos_ia)
    # MODIFICADO:
    # Debo adaptar campos por incorporación de método rank_cero_fijo
    campos_ia_mod <- adaptar_campos_ia_rank(
      campos_ia,
      campos_monetarios
    )

    # Verificación

    faltantes <- setdiff(campos_ia_mod, colnames(dataset))

    if (length(faltantes) > 0) {
      stop(
        "Hay variables seleccionadas por IA que no existen en dataset: ",
        paste(faltantes, collapse = ", ")
      )
    }


    cat("\n========================================\n")
    cat("ADAPTACION DE VARIABLES IA A RANK_ZERO_FIJO\n")
    cat("========================================\n")
    
    comparacion_ia <- data.table(
      original = campos_ia,
      adaptado = campos_ia_mod
    )
    
    cat("Variables adaptadas:", sum(comparacion_ia$original != comparacion_ia$adaptado), "\n")
    cat("Variables sin cambios:", sum(comparacion_ia$original == comparacion_ia$adaptado), "\n\n")
    
    print(
      comparacion_ia[original != adaptado]
    )
    
    cat("\n========================================\n")

    
    campos_buenos <- campos_ia_mod
    
    campos_control <- c(
      "numero_de_cliente",
      "foto_mes",
      "clase_ternaria",
      "clase01",
      "fold_train"
    )

    campos_conservar <- unique(
      c(campos_control, campos_buenos)
    )

    dataset <- dataset[
      ,
      campos_conservar,
      with = FALSE
    ]

    cat(
      "Variables seleccionadas por IA:",
      length(campos_buenos),
      "\n"
    )

    cat(
      "Dimensiones dataset reducido:",
      dim(dataset),
      "\n"
    )

    flush.console()

    # ----------------------------------------------------------
    # Variables que utilizará el modelo final
    # ----------------------------------------------------------


    # 7) Correr el modelo

    ### DEFINIMOS Y CORREMOS EL MODELO ###

    # entrenamiento
    dtrain <- lgb.Dataset(
      data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
      label= dataset[fold_train == TRUE, clase01],
      free_raw_data= TRUE
    )

    # datos de validation
    dvalidate <- lgb.Dataset(
      data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
      label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
      free_raw_data= TRUE
    )

    # LightGBM
    # parametros fijos del LightGBM
    PARAM$lgbm$param_fijos <- list(
      objective= "binary",
      metric= "auc",
      first_metric_only= TRUE,
      boost_from_average= TRUE,
      feature_pre_filter= FALSE,
      verbosity= -100,
      force_row_wise= TRUE, # para evitar warning
      seed= PARAM$semilla_primigenia,
      max_bin= 31,
      learning_rate= 0.03,
      feature_fraction= 0.5,
      num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
      early_stopping_rounds= 200,
      num_leaves= 64,
      min_data_in_leaf= 128
    )

    # Usamos mejores HP hallados en baseline
    PARAM$out$lgbm$mejores_hiperparametros <- list(
      num_leaves = 64,
      min_data_in_leaf = 128,
      feature_fraction = 0.5
    )

    PARAM$trainingstrategy$final_train <- c(
      201901, 201902, 201903, 201904, 201905, 201906,
      201907, 201908, 201909, 201910, 201911, 201912,
      202001, 202002, 202101, 202102, 202103, 202104,
      202105, 202106, 202107
    )


    dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

    # creo el dfinal_train en formato  LightGBM
    dfinal_train <- lgb.Dataset(
      data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
      label= dataset[fold_final_train == TRUE, clase01],
      free_raw_data= TRUE
    )

    # uno los parametros fijos y los mejores encontrados de los variables
    fijos <- copy(PARAM$lgbm$param_fijos)

    # quito lo que optimice en la Bayesian Optimization
    fijos$num_iterations <- NULL
    fijos$early_stopping_rounds <- NULL

    # agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
    param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

    cat("antes del train de final model:\n ")
    flush.console()
    final_model <- lgb.train(
      data= dfinal_train,
      param= param_final,
      verbose= -100
    )
    # grabo a disco el modelo en un formato para seres humanos
    lgb.save(final_model, paste0("modelo.txt", PARAM$semilla_primigenia))
    # ahora imprimo la importancia de variables

    tb_importancia <- as.data.table(lgb.importance(final_model))
    archivo_importancia <- paste0("impo.txt", PARAM$semilla_primigenia)

    fwrite( tb_importancia,
      file= archivo_importancia,
      sep= "\t"
    )

    # predicción
    PARAM$trainingstrategy$future <- c(202109)
    dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]
    # aplico final_model   a dfuture

    prediccion <- predict(
      final_model,
      data.matrix(dfuture[, campos_buenos, with= FALSE])
    )

    # Tabla de prediccion
    tb_prediccion <- dfuture[, list(numero_de_cliente, clase_ternaria)]
    tb_prediccion[, prob := prediccion]

    # grabo las probabilidad del modelo
    #  me va a ser util para hacer Ensembles de modelos
    fwrite(tb_prediccion,
      file= paste0("prediccion.txt", PARAM$semilla_primigenia),
      sep= "\t"
    )

}

In [13]:
# ============================================================
# RECUPERAR / COMPLETAR PREDICCIONES
# ============================================================

semillas <- c(
  # Semillas originales
  291173,
  400123,
  500029,
  689167,
  811501,
  317537,
  426389,
  573749,
  684769,
  918763,
  103801,
  214651,
  325537,
  437459,
  548621,
  659947,
  712687,
  763181,
  824831,
  856621,
  887789,
  931241,
  947317,
  967559,
  981799,
  100003,
  125003,
  150011,
  175013,
  200017,
  225023,
  250027,
  275027,
  300043,
  325043,
  350087,
  375091,
  400093,
  425101,
  450101,
  475103,
  500107,
  525127,
  550127,
  575129,
  600167,
  625169,
  650179,
  675179,
  700199,

  # 50 semillas nuevas
  100019,
  118369,
  136733,
  155087,
  173473,
  191837,
  210209,
  228577,
  246937,
  265313,
  283669,
  302053,
  320401,
  338777,
  357139,
  375509,
  393871,
  412249,
  430603,
  448969,
  467353,
  485717,
  504073,
  522449,
  540809,
  559183,
  577547,
  595927,
  614291,
  632647,
  651019,
  669391,
  687749,
  706117,
  724487,
  742891,
  761227,
  779591,
  797957,
  816329,
  834703,
  853057,
  871439,
  889783,
  908153,
  926533,
  944897,
  963253,
  981637,
  999983
)


# ============================================================
# 1. DETECTAR PREDICCIONES YA GUARDADAS
# ============================================================

semillas_guardadas <- semillas[
  file.exists(
    paste0(
      "prediccion.txt",
      semillas
    )
  )
]

semillas_faltantes <- setdiff(
  semillas,
  semillas_guardadas
)


cat(
  "Semillas ya guardadas:",
  paste(semillas_guardadas, collapse = ", "),
  "\n"
)

cat(
  "Semillas faltantes:",
  paste(semillas_faltantes, collapse = ", "),
  "\n"
)


# ============================================================
# 2. EJECUTAR SOLAMENTE LAS FALTANTES
# ============================================================

if (length(semillas_faltantes) > 0) {

  cat(
    "\nEjecutando",
    length(semillas_faltantes),
    "semillas faltantes...\n"
  )

  flush.console()

  for (semilla in semillas_faltantes) {

    cat(
      "\n========================================\n",
      "EJECUTANDO SEMILLA:",
      semilla,
      "\n========================================\n"
    )

    flush.console()

    run_experimento(semilla)

    cat(
      "\nFINALIZADA SEMILLA:",
      semilla,
      "\n"
    )

    flush.console()
  }

} else {

  cat(
    "\nNo hay semillas faltantes.\n"
  )
}

Semillas ya guardadas: 291173, 400123, 500029, 689167, 811501, 317537, 426389, 573749, 684769, 918763, 103801, 214651, 325537, 437459, 548621, 659947, 712687, 763181, 824831, 856621, 887789, 931241, 947317, 967559, 981799, 100003, 125003, 150011, 175013, 200017, 225023, 250027, 275027, 300043, 325043, 350087, 375091, 400093, 425101, 450101, 475103, 500107, 525127, 550127, 575129, 600167, 625169, 650179, 675179, 700199 
Semillas faltantes: 100019, 118369, 136733, 155087, 173473, 191837, 210209, 228577, 246937, 265313, 283669, 302053, 320401, 338777, 357139, 375509, 393871, 412249, 430603, 448969, 467353, 485717, 504073, 522449, 540809, 559183, 577547, 595927, 614291, 632647, 651019, 669391, 687749, 706117, 724487, 742891, 761227, 779591, 797957, 816329, 834703, 853057, 871439, 889783, 908153, 926533, 944897, 963253, 981637, 999983 

Ejecutando 50 semillas faltantes...

 EJECUTANDO SEMILLA: 100019 
2026-08-31 19:29:36  INICIO SEMILLA  100019 
inicio Corregir_Rotas()
fin Corregir_rotas()


In [14]:
cat("Listo")

Listo